In [3]:
%run cochain_complex.ipynb

In [265]:
class RegularCartanGeometry(object):
    def __init__(self,g,curv_str='K',coords=None):
        """Constructs a CartanGeometry of with the given curvature and symbol
        
        Arguments:
            * 'g' - a graded Lie algebra
            * 'c' - a 2-cochain from the complex C_+(g,g)

        Only intended for use in the computation of the Bianchi identity
        """
        self.symbol=g
        if coords==None:
            p=len(g.basis)
            self.coords=symbols('x_0:{}'.format(p))
        self.ndo_dict={}
        self.curvature=g.cochain_complex.regular_normal_2_cochain(curv_str)
        self.Bianchi_cache={}

        self.fund_der_cache=[]
        self.fund_invars=[]
        self.der_cache={}
    
    def queryJacobi(self):
        """Returns True if the Jacobi identity holds for the fundamental vector fields"""
        return NotImplemented
    
    def fund_der(self,f,i):
        """Returns the fundamental derivative in direction i of the function f
        
        Arguments:
            * 'f' - a rational function in terms of 2-tensors
            * 'ind' - an integer index for a direction
        """
        if type(f)==T_symb_elt:
            return f.parent.elt([self.fund_der(fj,i) for fj in f.vec])
        self.init_fund_der(i)
        return self.fund_der_cache[i].apply(f,normal=True)

    def init_fund_der(self,i):
        """Caches fundamental derivative DiffOpDicts up to index i"""
        if len(self.fund_der_cache)>i: return None
        # print('Initializing fund_der',i)
        self.compute_fund_der(i)
        return None
    
    def compute_fund_der(self,i):
        for j in range(len(self.fund_der_cache),i+1):
            self.fund_der_cache.append(DiffOpDict({(j,):1},self).normal_form())
        return None
    
    def update_fund_ders(self,ds_dict,distr):
        for k in range(len(self.fund_der_cache)):
            for t in self.fund_der_cache[k].d:
                self.fund_der_cache[k].d[t]=ds_subs(self.fund_der_cache[k].d[t],ds_dict,distr)
        return None

In [267]:
class DiffOpDict:
    """A DiffOpDict object represents a differential operator which is a 
    composition of fundamental derivatives with coefficients from the curvature tensor
    of the RegularCartanGeometry and its derivatives
    """

    def __init__(self,d,P):
        """Initializes a DiffOpDict from the dictionary d on the RegularCartanGeometry P.
        The keys of d should be tuples of indices, indicating fundamental derivatives; the values
        of d are the corresponding coefficients."""
        self.geom=P
        self.d=d

    def __repr__(self):
        return str(self.d)
    
    def __str__(self):
        return str(self.d)

    def __add__(self,other):
        # Perhaps parent check should be added to these methods
        d={}
        for k in set(self.d.keys()).union(set(other.d.keys())):
            d[k]=0
            if k in other.d: d[k]+=other.d[k]
            if k in self.d: d[k]+=self.d[k]
        return DiffOpDict(d,self.geom)
    
    def __mul__(self,other):
        if type(other)==DiffOpDict:
            r=DiffOpDict({},self.geom)
            for t1 in self.d:
                t1_term=DiffOpDict(other.d,self.geom)
                for i in reversed(t1):
                    temp=DiffOpDict({},self.geom)
                    for t2 in t1_term.d:
                        coeff=DiffOpDict({(i,):1},self.geom).apply(t1_term.d[t2])
                        temp+=DiffOpDict({(i,)+t2:t1_term.d[t2]},self.geom)
                        if coeff!=0: temp+=DiffOpDict({t2:coeff},self.geom)
                    t1_term=temp
                r+=self.d[t1]*t1_term
            return r
        else:
            r={}
            for k in self.d:
                r[k]=other*self.d[k]
            return DiffOpDict(r,self.geom)
    
    def __rmul__(self,other):
        # We shouldn't arrive here if type(other)==DiffOpDict
        return self*other
    
    def __neg__(self):
        d={}
        for k in self.d:
            d[k]=-self.d[k]
        return DiffOpDict(d,self.geom)
    
    def __sub__(self,other):
        return self+(-other)
    
    def __eq__(self,other):
        if type(other)==DiffOpDict:
            z=(self-other)
            z.clear_zeros()
            return z==0
        self.clear_zeros()
        if set(self.d.keys())=={tuple()}: return other==self.d[tuple()]
        if set(self.d.keys())==set(): return other==0
        return False

    def clear_zeros(self):
        for k in set(self.d.keys()):
            if self.d[k]==0:
                self.d.pop(k)
    
    def weakly_normalize_tuple(self,t):
        """Returns a weakly normal DiffOpDict which is equivalent to t. Weakly normal 
        means only indices of weight greater than -2 are included in self"""
        B=self.geom.symbol.basis
        a=None
        for i in reversed(t):
            if B[i].wght<-1: a=i
        if a==None: return DiffOpDict({t:1},self.geom)
        
        # Find Xi,Xj so that [Xi,Xj]=B[a].
        Xi=None
        Xj=None
        for i in range(len(B)):
            for j in range(i,len(B)):
                if B[i].ad(B[j])==B[a]: Xi,Xj=[B[i],B[j]]
        i,j=[B.index(Xi),B.index(Xj)]

        pre=DiffOpDict({t[0:t.index(a)]:1},self.geom)
        post=DiffOpDict({t[t.index(a)+1:len(t)]:1},self.geom)
        r=pre*DiffOpDict({(i,j):1},self.geom)*post
        r+=-pre*DiffOpDict({(j,i):1},self.geom)*post

        w=Xi.cast_as_ext_elt().wedge(Xj.cast_as_ext_elt())
        im_elt=self.geom.curvature.apply_cochain_map(w)
        for k in range(len(B)):
            if im_elt.vec[k]!=0:
                r+=(pre*DiffOpDict({(k,):im_elt.vec[k]},self.geom)*post)
        return r.weakly_normal_form()

    def strongly_normalize_tuple(self,t):
        """Returns a normal DiffOpDict which is equivalent to t. Strongly normal means 
        only indices of weight greater than -2 are included in self, and that the
        nonnegative operators appear first"""
        if len(t)==1: return DiffOpDict({t:1},self.geom)
        B=self.geom.symbol.basis
        a=None
        for i in range(len(t)-1):
            if B[t[i]].wght>=0 and B[t[i+1]].wght<0: a=i
        if a==None: return DiffOpDict({t:1},self.geom)

        pre=DiffOpDict({t[0:a]:1},self.geom)
        post=DiffOpDict({t[a+2:len(t)]:1},self.geom)
        r=pre*DiffOpDict({(t[a+1],t[a]):1},self.geom)*post
        im_elt=B[t[a]].ad(B[t[a+1]])
        for i in range(len(B)):
            if im_elt.vec[i]!=0:
                r+=pre*DiffOpDict({(i,):im_elt.vec[i]},self.geom)*post
        return r.normal_form()
    
    def weakly_normal_form(self):
        """Returns the weakly normal form of self"""
        #if self.query("WeaklyNormal"): return DiffOpDict(self.d,self.geom)
        r=DiffOpDict({},self.geom)
        for t in self.d:
            r+=self.d[t]*self.weakly_normalize_tuple(t)
        return r
    
    def normal_form(self):
        """Returns the normal form of self"""
        #if self.query("Normal"): return DiffOpDict(self.d,self.geom)
        w=DiffOpDict({},self.geom)
        for k in self.d:
            w+=self.weakly_normalize_tuple(k)*self.d[k]
        nf=DiffOpDict({},self.geom)
        for k in w.d:
            nf+=self.strongly_normalize_tuple(k)*w.d[k]
        return nf
    
    def apply(self,f,normal=False):
        """Applies the operator represented by self to the function f, which should be an 
        elementary function in terms of the curvatures of self.geom"""
        if type(f)==T_symb_elt:
            return f.parent.elt([self.apply(a) for a in f.vec])
        if normal: nf=self
        else: nf=self.normal_form()
        r=0
        for k in nf.d:
            temp=f
            for i in reversed(k):
                temp=self.normal_ind_der(temp,i)
            r+=temp*nf.d[k]
        return r

    def normal_ind_der(self,f,i):
        """Applies the fundamental derivative corresponding to i to the function
        
        Arguments:
            * 'i' - an integer corresponding to a fundamental derivative of weight >=-1
            * 'f' - an elementary function in terms of the curvatures of self.geom"""
        
        if isinstance(f,numbers.Number):
            return 0
        if type(f)==Add:
            return Add(*[self.normal_ind_der(A,i) for A in f.args])
        if type(f)==Mul:
            result=0
            for j in range(len(f.args)):
                result+=self.normal_ind_der(f.args[j],i)*Mul(*list(f.args[0:j]+f.args[j+1:len(f.args)]))
            return result
        if type(f)==Pow:
            return f.exp*f.base**(f.exp-1)*self.normal_ind_der(f.base,i)
        if type(f)==Symbol: return 0
        if type(f)==exp:
            return f*self.normal_ind_der(f.args[0],i)
        if type(f)==Indexed:
            if self.geom.symbol.basis[i].wght<0:
                return f.base[f.indices+(i,)]
            else:
                # nonnegative indices act via (-ad) after normalizing
                if len(f.indices)==3:
                    g=self.geom.symbol
                    Xi=g.basis[i]
                    Xi_curv=(-Xi).ad(self.geom.curvature,mod='CE')
                    dwi=g.cochain_complex.dwi(tuple([g.basis_strs[a] for a in f.indices]))
                    try: return Xi_curv.vd[dwi[0]][dwi[1]][dwi[2]]
                    except: return 0
                else:
                    t=(i,)+tuple(reversed(f.indices[3:len(f.indices)]))
                    if not f.base[*(f.indices+(i,))] in self.geom.der_cache: 
                        d=DiffOpDict({t:1},self.geom)
                        self.geom.der_cache[f.base[*(f.indices+(i,))]]=d.apply(f.base[*f.indices[0:3]])
                    return self.geom.der_cache[f.base[*(f.indices+(i,))]]
    
    def query(self,property):
        """Queries if self satisfies property among "WeaklyNormal" and "Normal"
        """
        if property=="WeaklyNormal":
            for k in self.d:
                for i in k:
                    if self.geom.symbol.basis[i].wght<-1: return False
            return True
        if property=="Normal":
            if not self.query("WeaklyNormal"): 
                return False
            for k in self.d:
                a=len(k)
                for i in range(len(k)):
                    if self.geom.symbol.basis[len(k)-1-i].wght>-1: a=len(k)-1-i
                for i in range(a+1,len(k)):
                    if self.geom.symbol.basis[k[i]].wght==-1: 
                        return False
            return True